# 04-8. 임시 파일과 원자적 저장

## Goal

완성되지 않은 JSON이 최종 경로에 노출되지 않도록 같은 디렉터리의 고유한 임시 파일에 먼저 씁니다. 성공과 실패를 모두 재현하며 다음 계약을 검증합니다.

- 저장 전에 보고서의 최소 구조를 검증합니다.
- 쓰기·버퍼 반영·재읽기·구조 검증 뒤 `os.replace()`로 교체합니다.
- 직렬화 또는 검증 실패 시 기존 정상 결과를 보존하고 임시 파일을 정리합니다.
- 입력과 출력이 같은 파일을 가리키는 실수를 거부합니다.

## Setup

모든 파일은 `TemporaryDirectory` 안에서만 생성합니다. 기존 프로젝트 파일이나 개인 문서는 변경하지 않습니다. Python 3.10 이상과 표준 라이브러리만 사용합니다.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from tempfile import NamedTemporaryFile, TemporaryDirectory

assert sys.version_info >= (3, 10), "Python 3.10 이상이 필요합니다"

_lab = TemporaryDirectory(prefix="chapter-04-8-")
LAB_DIR = Path(_lab.name).resolve()
OUTPUT_PATH = LAB_DIR / "reports" / "report.json"

print("임시 실습 디렉터리:", LAB_DIR)

## Steps

### 1. 저장할 보고서의 구조 검증하기

JSON으로 직렬화할 수 있다는 사실과 업무 계약을 만족한다는 사실은 다릅니다. 이 예제는 최상위 객체와 `summary` 객체를 최소 계약으로 사용합니다.

In [ ]:
def validate_report(value: object) -> None:
    if not isinstance(value, dict):
        raise TypeError("보고서 최상위 값은 JSON 객체여야 합니다")
    if "summary" not in value:
        raise ValueError("summary 필드가 없습니다")
    if not isinstance(value["summary"], dict):
        raise TypeError("summary는 JSON 객체여야 합니다")


valid_example = {"summary": {"valid": 2, "errors": 1}}
validate_report(valid_example)
print("보고서 구조 검증 통과")

### 2. 임시 파일을 검증한 뒤 최종 경로로 교체하기

임시 파일은 최종 출력과 같은 디렉터리에 만듭니다. 파일을 연 직후 경로를 기록하고, 어떤 단계에서 실패하더라도 `finally`에서 남은 임시 파일을 제거합니다.

In [ ]:
def save_json_safely(data, output_path: Path, validator=None) -> Path:
    output_path = Path(output_path).expanduser().resolve()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = None

    try:
        with NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            dir=output_path.parent,
            prefix=f".{output_path.name}.",
            suffix=".tmp",
            delete=False,
        ) as temporary:
            temporary_path = Path(temporary.name)
            json.dump(
                data,
                temporary,
                ensure_ascii=False,
                indent=2,
                allow_nan=False,
            )
            temporary.write("\n")
            temporary.flush()
            os.fsync(temporary.fileno())

        with temporary_path.open("r", encoding="utf-8") as file:
            restored = json.load(file)

        if validator is not None:
            validator(restored)

        os.replace(temporary_path, output_path)
        temporary_path = None
        return output_path
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)

### 3. 정상 보고서 저장하고 다시 읽기

출력 부모 디렉터리가 없어도 함수가 생성합니다. 반환 경로와 다시 읽은 JSON을 함께 확인합니다.

In [ ]:
first_report = {
    "summary": {"valid": 2, "errors": 1},
    "message": "첫 번째 정상 보고서",
}
saved_path = save_json_safely(first_report, OUTPUT_PATH, validate_report)

with saved_path.open("r", encoding="utf-8") as file:
    restored_report = json.load(file)

print("저장 경로:", saved_path)
print(json.dumps(restored_report, ensure_ascii=False, indent=2))

### 4. 직렬화와 구조 검증 실패 재현하기

직렬화할 수 없는 객체와 잘못된 `summary` 자료형을 차례로 저장해 봅니다. 두 시도 모두 실패해야 하며, 최종 경로에는 첫 번째 정상 보고서가 그대로 남아야 합니다.

In [ ]:
try:
    save_json_safely(
        {"summary": {"valid": 3}, "not_json": object()},
        OUTPUT_PATH,
        validate_report,
    )
except TypeError as exc:
    serialization_error = str(exc)
else:
    raise AssertionError("직렬화 실패가 발생해야 합니다")

try:
    save_json_safely(
        {"summary": ["잘못된 자료형"]},
        OUTPUT_PATH,
        validate_report,
    )
except TypeError as exc:
    validation_error = str(exc)
else:
    raise AssertionError("구조 검증 실패가 발생해야 합니다")

with OUTPUT_PATH.open("r", encoding="utf-8") as file:
    report_after_failures = json.load(file)

temporary_files_after_failures = list(
    OUTPUT_PATH.parent.glob(f".{OUTPUT_PATH.name}.*.tmp")
)
print("직렬화 오류:", serialization_error)
print("검증 오류:", validation_error)

### 5. 입력과 출력 경로 충돌 거부하기

경로 문자열을 정규화해 비교하고, 두 경로가 이미 존재하면 `samefile()`로 같은 파일인지 한 번 더 확인합니다. 이 검사는 실수를 줄이지만 검사 직후 경로가 바뀌는 경쟁 조건까지 제거하지는 않습니다.

In [ ]:
def ensure_different_paths(input_path: Path, output_path: Path) -> None:
    source = Path(input_path).expanduser().resolve()
    destination = Path(output_path).expanduser().resolve()

    if source == destination:
        raise ValueError("입력과 출력 경로는 달라야 합니다")
    if source.exists() and destination.exists() and source.samefile(destination):
        raise ValueError("입력과 출력이 같은 파일을 가리킵니다")


INPUT_PATH = LAB_DIR / "input.txt"
INPUT_PATH.write_text("원본 입력\n", encoding="utf-8")
ensure_different_paths(INPUT_PATH, OUTPUT_PATH)

try:
    ensure_different_paths(INPUT_PATH, INPUT_PATH)
except ValueError as exc:
    collision_error = str(exc)
else:
    raise AssertionError("같은 입력·출력 경로를 거부해야 합니다")

print("경로 충돌 오류:", collision_error)

## Checks

정상 결과의 재읽기, 두 실패 뒤의 기존 결과 보존, 임시 파일 정리, 경로 충돌 거부를 한 번에 확인합니다.

In [ ]:
assert saved_path == OUTPUT_PATH.resolve()
assert restored_report == first_report
assert report_after_failures == first_report
assert temporary_files_after_failures == []
assert "JSON serializable" in serialization_error
assert "summary" in validation_error
assert "달라야" in collision_error
assert OUTPUT_PATH.read_bytes().endswith(b"\n")
assert INPUT_PATH.read_text(encoding="utf-8") == "원본 입력\n"

print("04-8 계약 검증 통과")

### 원자적 교체가 보장하지 않는 것

`os.replace()`는 일반적인 같은 로컬 파일 시스템에서 독자가 이전 파일 또는 새 파일 중 하나를 보게 하는 이름 교체를 제공합니다. 하지만 백업, 여러 작성자의 순서, 모든 네트워크 파일 시스템의 동작, 파일 권한 유지, 정전 뒤의 완전한 내구성을 자동으로 보장하지는 않습니다.

In [ ]:
temporary_root = LAB_DIR
_lab.cleanup()
assert not temporary_root.exists()
print("임시 실습 디렉터리 정리 완료")

## Next Steps

- 04-7의 JSON Lines 출력 대상을 최종 파일이 아니라 이 절의 임시 파일 객체로 연결합니다.
- 서비스 요구사항에 따라 백업 이름·보존 기간·동시 쓰기 정책을 별도로 정의합니다.
- 다음 절에서는 파일 분석 보고서를 같은 절차로 저장하고 원본 해시가 바뀌지 않았는지 검증합니다.